# Bronze Layer Ingestion

## Objective

Ingest raw customer, product, order, and transaction data into
Bronze Parquet datasets using PySpark.

Bronze preserves source-level information with minimal transformation.

Business cleaning and validation will be performed in the Silver layer.

Configuration / Paths

In [35]:
RAW_PATH = "/content/pyspark-ecommerce-analytics/data/raw"
BRONZE_PATH = "/content/pyspark-ecommerce-analytics/data/bronze"

CUSTOMERS_RAW = f"{RAW_PATH}/customers.csv"
PRODUCTS_RAW = f"{RAW_PATH}/products.json"
ORDERS_RAW = f"{RAW_PATH}/orders.csv"
TRANSACTIONS_RAW = f"{RAW_PATH}/transactions.csv"

Spark Session

In [36]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName('Bronze-ingestion').getOrCreate()
spark.version

'4.0.4'

## **Customer**

Read Customer Data

In [37]:
df_customer_bronze = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(CUSTOMERS_RAW)

df_customer_bronze.printSchema()
df_customer_bronze.show(10)

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)

+-----------+----------+---------+--------------------+--------------+--------+-----------+----------------+--------------+
|customer_id|first_name|last_name|               email|       country|    city|signup_date|customer_segment|account_status|
+-----------+----------+---------+--------------------+--------------+--------+-----------+----------------+--------------+
|    C000001|       Mia|    Davis|mia.davis1@exampl...|       Germany|  Munich| 2024-10-14|  Small Business|        Active|
|    C000002|     Arjun|    Davis|arjun.davis2@exam...|        Canada| Calgary| 2025-08-11|        Consumer|        Active|

Add Ingestion Metadata (Customer)

In [38]:
df_customer_bronze = df_customer_bronze.withColumn(
    '_ingestion_timestamp',
    F.current_timestamp()
)

df_customer_bronze = df_customer_bronze.withColumn(
    '_source_file',
    F.input_file_name()
)

df_customer_bronze.show(5, truncate=False)

+-----------+----------+---------+-------------------------+-------------+-------+-----------+----------------+--------------+--------------------------+------------------------------------------------------------------+
|customer_id|first_name|last_name|email                    |country      |city   |signup_date|customer_segment|account_status|_ingestion_timestamp      |_source_file                                                      |
+-----------+----------+---------+-------------------------+-------------+-------+-----------+----------------+--------------+--------------------------+------------------------------------------------------------------+
|C000001    |Mia       |Davis    |mia.davis1@example.com   |Germany      |Munich |2024-10-14 |Small Business  |Active        |2026-10-01 13:50:17.461667|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
|C000002    |Arjun     |Davis    |arjun.davis2@example.com |Canada       |Calgary|2025-08-11 |Consumer        |Activ

Write data to Bronze (Customer)

In [39]:
df_customer_bronze.write.mode('overwrite').parquet(f"{BRONZE_PATH}/customers")

Read Back and Verify (Customer)

In [40]:
bronze_customer = spark.read.parquet(f"{BRONZE_PATH}/customers")
bronze_customer.printSchema()
bronze_customer.show(5, truncate=False)
print('Row Count')
print('---------')
bronze_customer.count()


root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+-----------+----------+---------+-------------------------+-------------+-------+-----------+----------------+--------------+--------------------------+------------------------------------------------------------------+
|customer_id|first_name|last_name|email                    |country      |city   |signup_date|customer_segment|account_status|_ingestion_timestamp      |_source_file                                                      |
+-----------+----------+---------+-------------------------+-------------+----

10000

## **Orders**

Read Orders Data

In [41]:
df_orders_bronze = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(ORDERS_RAW)

df_orders_bronze.printSchema()
df_orders_bronze.show(10)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: string (nullable = true)

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+
|O0000001|    C002471|2024-10-27|   Completed|          Card|       Australia|    Melbourne|    1510.72|
|O0000002|    C009466|2024-04-16|   Completed|           UPI|  United Kingdom|   Manchester|    1716.70|
|O0000003|    C002882|2025-02-19|   Completed|           UPI|          Canada|     Montreal|    1105.60|
|O0000004|   

Add Ingestion Metadata (Orders)

In [42]:
df_orders_bronze = df_orders_bronze.withColumn(
    '_ingestion_timestamp',
    F.current_timestamp()
)

df_orders_bronze = df_orders_bronze.withColumn(
    '_source_file',
    F.input_file_name()
)

df_orders_bronze.show(5, truncate=False)

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+---------------------------------------------------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|_ingestion_timestamp      |_source_file                                                   |
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+---------------------------------------------------------------+
|O0000001|C002471    |2024-10-27|Completed   |Card          |Australia       |Melbourne    |1510.72    |2026-10-01 13:50:19.051145|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|O0000002|C009466    |2024-04-16|Completed   |UPI           |United Kingdom  |Manchester   |1716.70    |2026-10-01 13:50:19.051145|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|O0000003|C002882   

Write data to Bronze (Orders)

In [43]:
df_orders_bronze.write.mode('overwrite').parquet(f"{BRONZE_PATH}/orders")

Read Back and Verify (Orders)

In [44]:
bronze_orders = spark.read.parquet(f"{BRONZE_PATH}/orders")
bronze_orders.printSchema()
bronze_orders.show(5, truncate=False)
print('Row Count')
print('---------')
bronze_orders.count()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- order_total: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+---------------------------------------------------------------+
|order_id|customer_id|order_date|order_status|payment_method|shipping_country|shipping_city|order_total|_ingestion_timestamp      |_source_file                                                   |
+--------+-----------+----------+------------+--------------+----------------+-------------+-----------+--------------------------+---------------------

30000

## **Products**

Read Products Data

In [45]:
df_products_bronze = spark.read.format('json').option('inferSchema', 'true').load(PRODUCTS_RAW)
df_products_bronze.printSchema()
df_products_bronze.show(5, truncate=False)

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)

+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+
|brand |category   |cost_price|created_date|is_active|product_id|product_name       |subcategory|unit_price|
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+
|Nova  |Beauty     |14.01     |2025-05-01  |true     |P00001    |Orion Wellness 0001|Wellness   |22.83     |
|Orion |Electronics|140.59    |2024-09-06  |true     |P00002    |Atlas Laptops 0002 |Laptops    |253.71    |
|Harbor|Beauty     |27.95     |2024-03-07  |true     |P00003    |Cedar Wel

Add Ingestion Metadata (Products)

In [46]:
df_products_bronze = df_products_bronze.withColumn(
    '_ingestion_timestamp',
    F.current_timestamp()
)

df_products_bronze = df_products_bronze.withColumn(
    '_source_file',
    F.input_file_name()
)

df_products_bronze.show(5, truncate=False)

+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+--------------------------+------------------------------------------------------------------+
|brand |category   |cost_price|created_date|is_active|product_id|product_name       |subcategory|unit_price|_ingestion_timestamp      |_source_file                                                      |
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+--------------------------+------------------------------------------------------------------+
|Nova  |Beauty     |14.01     |2025-05-01  |true     |P00001    |Orion Wellness 0001|Wellness   |22.83     |2026-10-01 13:50:20.195339|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
|Orion |Electronics|140.59    |2024-09-06  |true     |P00002    |Atlas Laptops 0002 |Laptops    |253.71    |2026-10-01 13:50:20.195339|file:///content/pyspark-ecommerce-analytics/data/raw/

Write data to Bronze (Products)

In [47]:
df_products_bronze.write.mode('overwrite').parquet(f"{BRONZE_PATH}/products")

Read Back and Verify (Products)

In [48]:
bronze_products = spark.read.parquet(f"{BRONZE_PATH}/products")
bronze_products.printSchema()
bronze_products.show(5, truncate=False)
print('Row Count')
print('---------')
bronze_products.count()

root
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- created_date: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+-------------------------+------------------------------------------------------------------+
|brand |category   |cost_price|created_date|is_active|product_id|product_name       |subcategory|unit_price|_ingestion_timestamp     |_source_file                                                      |
+------+-----------+----------+------------+---------+----------+-------------------+-----------+----------+--------

1000

## **Transactions**

Read Transactions Data

In [49]:
df_transactions_bronze = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(TRANSACTIONS_RAW)
df_transactions_bronze.printSchema()
df_transactions_bronze.show(5, truncate=False)

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- payment_status: string (nullable = true)

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+
|T00000001     |O0028299|P00169    |2024-04-04      |2       |82.22     |16.44          |11.84     |159.84            |Captured      |
|T00000002 

Add Ingestion Metadata (Transactions)

In [50]:
df_transactions_bronze = df_transactions_bronze.withColumn(
    '_ingestion_timestamp',
    F.current_timestamp()
)

df_transactions_bronze = df_transactions_bronze.withColumn(
    '_source_file',
    F.input_file_name()
)

df_transactions_bronze.show(5, truncate=False)

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------------+---------------------------------------------------------------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|_ingestion_timestamp      |_source_file                                                         |
+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+--------------------------+---------------------------------------------------------------------+
|T00000001     |O0028299|P00169    |2024-04-04      |2       |82.22     |16.44          |11.84     |159.84            |Captured      |2026-10-01 13:50:21.292602|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
|T00000002     |O0017225|P00332    |2024-04-07      |5       |28.76     

Write data to Bronze (Transactions)

In [51]:
df_transactions_bronze.write.mode('overwrite').parquet(f"{BRONZE_PATH}/transactions")

Read Back and Verify (Transactions)

In [52]:
bronze_transactions = spark.read.parquet(f"{BRONZE_PATH}/transactions")
bronze_transactions.printSchema()
bronze_transactions.show(5, truncate=False)
print('Row Count')
print('---------')
bronze_transactions.count()

root
 |-- transaction_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- transaction_amount: double (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- _ingestion_timestamp: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)

+--------------+--------+----------+----------------+--------+----------+---------------+----------+------------------+--------------+-------------------------+---------------------------------------------------------------------+
|transaction_id|order_id|product_id|transaction_date|quantity|unit_price|discount_amount|tax_amount|transaction_amount|payment_status|_ingestion_timestamp     |_source_file                                                 

50000

## **Overview**

In [53]:
from pyspark.sql import Row
datasets = {
    "Customers": (df_customer_bronze.count(), bronze_customer.count()),
    "Products": (df_products_bronze.count(), bronze_products.count()),
    "Orders": (df_orders_bronze.count(), bronze_orders.count()),
    "Transactions": (df_transactions_bronze.count(), bronze_transactions.count())
}

overview = [
    Row(Dataset=name, Raw=raw_count, Bronze=bronze_count)
    for name, (raw_count, bronze_count) in datasets.items()
]

spark.createDataFrame(overview).show()

+------------+-----+------+
|     Dataset|  Raw|Bronze|
+------------+-----+------+
|   Customers|10000| 10000|
|    Products| 1000|  1000|
|      Orders|30000| 30000|
|Transactions|50000| 50000|
+------------+-----+------+



## Metadata Verifications

Customers

In [54]:
bronze_customer.select(
    "_ingestion_timestamp",
    "_source_file"
).show(5, truncate=False)

+--------------------------+------------------------------------------------------------------+
|_ingestion_timestamp      |_source_file                                                      |
+--------------------------+------------------------------------------------------------------+
|2026-10-01 13:50:17.563182|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
|2026-10-01 13:50:17.563182|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
|2026-10-01 13:50:17.563182|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
|2026-10-01 13:50:17.563182|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
|2026-10-01 13:50:17.563182|file:///content/pyspark-ecommerce-analytics/data/raw/customers.csv|
+--------------------------+------------------------------------------------------------------+
only showing top 5 rows


Orders

In [55]:
bronze_orders.select(
    "_ingestion_timestamp",
    "_source_file"
).show(5, truncate=False)

+--------------------------+---------------------------------------------------------------+
|_ingestion_timestamp      |_source_file                                                   |
+--------------------------+---------------------------------------------------------------+
|2026-10-01 13:50:19.144051|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|2026-10-01 13:50:19.144051|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|2026-10-01 13:50:19.144051|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|2026-10-01 13:50:19.144051|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
|2026-10-01 13:50:19.144051|file:///content/pyspark-ecommerce-analytics/data/raw/orders.csv|
+--------------------------+---------------------------------------------------------------+
only showing top 5 rows


Products

In [56]:
bronze_products.select(
    "_ingestion_timestamp",
    "_source_file"
).show(5, truncate=False)

+-------------------------+------------------------------------------------------------------+
|_ingestion_timestamp     |_source_file                                                      |
+-------------------------+------------------------------------------------------------------+
|2026-10-01 13:50:20.27222|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
|2026-10-01 13:50:20.27222|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
|2026-10-01 13:50:20.27222|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
|2026-10-01 13:50:20.27222|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
|2026-10-01 13:50:20.27222|file:///content/pyspark-ecommerce-analytics/data/raw/products.json|
+-------------------------+------------------------------------------------------------------+
only showing top 5 rows


Transactions

In [57]:
bronze_transactions.select(
    "_ingestion_timestamp",
    "_source_file"
).show(5, truncate=False)

+-------------------------+---------------------------------------------------------------------+
|_ingestion_timestamp     |_source_file                                                         |
+-------------------------+---------------------------------------------------------------------+
|2026-10-01 13:50:21.41399|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
|2026-10-01 13:50:21.41399|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
|2026-10-01 13:50:21.41399|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
|2026-10-01 13:50:21.41399|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
|2026-10-01 13:50:21.41399|file:///content/pyspark-ecommerce-analytics/data/raw/transactions.csv|
+-------------------------+---------------------------------------------------------------------+
only showing top 5 rows
